In [3]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ==============================================================================
# 1. CONFIGURACIÓN DEL DRIVER (El "Navegador Invisible" para Docker)
# ==============================================================================
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ==============================================================================
# 2. INICIALIZACIÓN DE VARIABLES (La Memoria)
# ==============================================================================
datos_totales = []
paginas_objetivo = 3 # Extraeremos el Top 300 de criptomonedas (100 por página)

try:
    print("Conectando a CoinGecko...")
    driver.get("https://www.coingecko.com/")
    
    # ==========================================================================
    # 3. BUCLE DE PAGINACIÓN (El Desafío de la Semana)
    # ==========================================================================
    for i in range(paginas_objetivo):
        print(f"\n--- Procesando Página {i + 1} ---")
        
        # Esperamos hasta 10 segundos para asegurarnos de que la tabla cargue
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table tbody tr"))
        )
        
        # CAPTURA: Buscamos todas las filas de la tabla
        filas = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
        print(f"Buscando en {len(filas)} filas...")
        
        for fila in filas:
            try:
                # Extraemos todas las celdas de la fila
                celdas = fila.find_elements(By.TAG_NAME, "td")
                
                # En CoinGecko: La columna 3 es el Nombre, la columna 5 suele ser el Precio
                if len(celdas) >= 5:
                    nombre_crypto = celdas[2].text.strip().split('\n')[0] # Limpiamos saltos de línea
                    precio_crypto = celdas[4].text.strip()
                    
                    if nombre_crypto and precio_crypto:
                        datos_totales.append({
                            "Pagina": i + 1,
                            "Criptomoneda": nombre_crypto,
                            "Precio": precio_crypto
                        })
            except Exception:
                continue # Si una fila es publicidad o falla, seguimos con la otra
                
        # NAVEGACIÓN: Lógica para saltar a la siguiente página
        try:
            pagina_siguiente = i + 2 # Si estamos en la 1 (i=0), buscamos la 2
            print(f"Buscando enlace hacia la página {pagina_siguiente}...")
            
            # Buscamos cualquier enlace (a) cuyo destino (href) termine en ?page=X
            selector_enlace = f"a[href$='?page={pagina_siguiente}']"
            
            # Usamos un WebDriverWait corto por si el botón tarda un segundo en renderizar
            boton_siguiente = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, selector_enlace))
            )
            
            # Ejecutamos el clic vía JavaScript para evitar banners
            driver.execute_script("arguments[0].click();", boton_siguiente)
            
            print("Clic exitoso. Esperando carga...")
            time.sleep(5) 
            
        except Exception as e:
            print(f"No se pudo avanzar a la página {pagina_siguiente}. Error: No se encontró el enlace.")
            break

    # ==========================================================================
    # 4. SALIDA DE DATOS
    # ==========================================================================
    if datos_totales:
        df = pd.DataFrame(datos_totales)
        df.to_csv("datos_coingecko.csv", index=False)
        print("\n¡Desafío completado con éxito! ")
        print(f"Total de registros capturados: {len(datos_totales)}")
        print("-" * 40)
        print(df.head()) # Muestra los primeros 5
        print("...")
        print(df.tail()) # Muestra los últimos 5
    else:
        print("\nNo se capturaron datos.")

except Exception as e:
    print(f"Error detectado durante la ejecución: {e}")

finally:
    # SIEMPRE cerrar el navegador para evitar procesos 'zombis' en Docker
    driver.quit()
    print("\nNavegador cerrado.")

Conectando a CoinGecko...

--- Procesando Página 1 ---
Buscando en 100 filas...
Buscando enlace hacia la página 2...
Clic exitoso. Esperando carga...

--- Procesando Página 2 ---
Buscando en 100 filas...
Buscando enlace hacia la página 3...
Clic exitoso. Esperando carga...

--- Procesando Página 3 ---
Buscando en 100 filas...
Buscando enlace hacia la página 4...
Clic exitoso. Esperando carga...

¡Desafío completado con éxito! 
Total de registros capturados: 300
----------------------------------------
   Pagina Criptomoneda      Precio
0       1      Bitcoin  $68,144.78
1       1     Ethereum   $2,101.98
2       1       Tether     $0.9991
3       1          BNB     $616.18
4       1          XRP       $1.35
...
     Pagina Criptomoneda     Precio
295       3      ApeCoin   $0.08598
296       3       Targon     $19.08
297       3       JupUSD      $1.00
298       3   Theta Fuel   $0.01132
299       3     Wiki Cat  $0.061522

Navegador cerrado.
